# Suite2p — batch processing

Load one settings file, then run the same settings on several recordings.

* `settings`: **how** to process (shared by all recordings).
* `db`: **what** to process (one per recording: data and save paths, planes, channels).

Before running, remember to set the conda/suite2p env as the python kernel.

In [1]:
import json
import logging
import shutil
from pathlib import Path
import suite2p
from suite2p import default_db
from suite2p.run_s2p import logger_setup, get_save_folder

## 1. Load the settings

In [2]:
settings_path = r"C:\Users\User\Documents\Code\suite2p\Python_demo_Federico\Dendrites_iGluSNFR4s_settings.json"

with open(settings_path) as f:
    settings = json.load(f)

## 2. One `db` per recording

In [ ]:
dbs = [
    {
        "data_path":  [
            r"Z:\Lab_Share\Federico\suite2p_samples\AL042\1",
        ],
            "fast_disk":  "C:\\",   # local disk for the temporary binary (a raw string can't end in a backslash)
        "save_path0": r"Z:\Lab_Share\Federico\suite2p_samples\processed\AL042",
    },
 
    {
        "data_path":  [r"Z:\Lab_Share\Federico\suite2p_samples\NM035\1"],
            "fast_disk":  "C:\\",   # local disk for the temporary binary (a raw string can't end in a backslash)
        "save_path0": r"Z:\Lab_Share\Federico\suite2p_samples\processed\NM035",
    },
]

## 3. Run

If a recording already has results, suite2p reuses the old `reg_outputs.npy` and **skips registration**, even when it has just rebuilt an unregistered binary from the TIFFs. To avoid processing an unaligned movie:

* `OVERWRITE = False`: recordings that already have results are skipped.
* `OVERWRITE = True`: that recording's `suite2p` folder is deleted and processed from scratch.

In [4]:
OVERWRITE = True

s2p_logger = logging.getLogger("suite2p")

for db in dbs:
    db = {**default_db(), **db}
    save_folder = Path(get_save_folder(db))

    if any(save_folder.glob("plane*/reg_outputs.npy")):
        if not OVERWRITE:
            print(f"SKIPPING (already processed, set OVERWRITE = True to redo): {save_folder}")
            continue
        print(f"Deleting previous results: {save_folder}")
        shutil.rmtree(save_folder)

    # run_s2p() doesn't set up logging itself (the CLI/GUI do), so without this
    # the per-stage messages are not shown. Drop the previous recording's log
    # file handler so each run.log only contains its own recording.
    for h in [h for h in s2p_logger.handlers if isinstance(h, logging.FileHandler)]:
        s2p_logger.removeHandler(h)
        h.close()
    logger_setup(str(save_folder))   # prints to this cell + writes <save_folder>/run.log

    suite2p.run_s2p(settings=settings, db=db)

2026-11-12 13:22:18,496 [INFO] 
cellpose version: 	1.0.0.1 
platform:       	win32 
python version: 	3.11.14 
torch version:  	2.10.0+cu126
2026-11-12 13:22:18,497 [INFO] data_path: ['Z:\\Lab_Share\\Federico\\suite2p_samples\\NM035\\1']
2026-11-12 13:22:18,521 [INFO] ** Found 2 files - converting to binary **


SKIPPING (already processed, set OVERWRITE = True to redo): Z:\Lab_Share\Federico\suite2p_samples\processed\AL042\suite2p
creating log file Z:\Lab_Share\Federico\suite2p_samples\processed\NM035\suite2p\run.log


2026-11-12 13:22:54,583 [INFO] 2000 frames of binary, time 36.02 sec.
2026-11-12 13:23:22,409 [INFO] 4000 frames of binary, time 63.85 sec.
2026-11-12 13:23:22,615 [INFO] Wrote 4000 frames per binary, 1 folders + 1 channels, 64.13sec
2026-11-12 13:23:22,663 [INFO] >>>>>>>>>>>>>>>>>>>>> PLANE 0 <<<<<<<<<<<<<<<<<<<<<<
2026-11-12 13:23:22,789 [INFO] ** torch.device('cuda') installed and working. **
2026-11-12 13:23:22,802 [INFO] Running registration
2026-11-12 13:23:22,833 [INFO] binary output path: Z:\Lab_Share\Federico\suite2p_samples\processed\NM035\suite2p\plane0\data.bin
2026-11-12 13:23:22,869 [INFO] NOTE: applying default classifier: C:\Users\User\.suite2p\classifiers\classifier_user.npy
2026-11-12 13:23:22,869 [INFO] ----------- REGISTRATION
2026-11-12 13:23:22,869 [INFO] registering 1 channels
2026-11-12 13:23:28,209 [INFO] Reference frame, 2.85 sec.
2026-11-12 13:23:28,251 [INFO] Registering 4000 frames in 40 batches
2026-11-12 13:23:52,390 [INFO] 100%|##########| 40/40 [00:24<0